# Агрегация жил кабеля — v4.0 (OR-Tools CP-SAT)

## Почему OR-Tools?
Предыдущие версии использовали **жадный алгоритм** — порядок обработки кабелей
влиял на результат и порождал ошибки. CP-SAT — это **точный оптимизатор**:
он перебирает все варианты и гарантирует математически оптимальное решение.

## Модель
**Переменная** `x[кабель, барабан]` = целое число метров с барабана  
(одинаково для ВСЕХ жил кабеля — физически корректно).

**Ограничения:**
- Сумма `x[k,d] × жил[k]` по всем кабелям ≤ ёмкость барабана d
- Сумма `x[k,d]` по барабанам ≥ `min_length[k]` (гибкий) или = (фиксированный)
- Каждый ненулевой сегмент ≥ `MIN_SEGMENT` м (физ. ограничение машины)

**Цель:** максимизировать использование провода, затем минимизировать смены барабанов.

## Маркировка
- `flexible=False` — **"сдача одной длиной"** — строго фиксировано
- `flexible=True`  — **"сдача макс. длиной"** — алгоритм удлинит для поглощения остатков

---

## Что нового в v4
| # | Изменение |
|---|-----------|
| 1 | **Группировка кабелей по составу жил** — FFD и наборы формируются раздельно для каждой группы цветов (5ж, 4ж, 3ж, 2ж). Метки: , . |
| 2 | **Автоматическое разрешение коллизий меток** — если два набора имеют одинаковое число жил, но разный состав цветов, они получают суффиксы: , . |
| 3 | **Порядок скрутки** = обратный порядку намотки (изолировщик мотает снизу вверх, скрутчик разматывает сверху вниз). |
| 4 | **Масштабируемый порядок цветов** — для ≤8 цветов перебор всех перестановок; для ≥9 цветов (пары 8×2, 16×2) — жадный алгоритм ближайшего соседа. |
| 5 | **Авто-палитра** — неизвестные цвета жил автоматически получают фоновый цвет в Excel (расширенный список: красный, белый, серый, оранжевый, фиолетовый…). |

---

## ГАЙД: КАК УПРАВЛЯТЬ АЛГОРИТМОМ

### 1. Исходные данные (Раздел 2)

| Что менять | Где | Эффект |
|---|---|---|
| `Drum(length=...)` | `cell-v3-drums` | Ёмкость барабана в метрах |
| `Cable(min_length=...)` | `cell-v3-cables` | Минимальная длина жилы |
| `flexible=True/False` | `cell-v3-cables` | True → кабель будет удлинён |

### 2. Настройки запуска (Раздел 3) — главные рычаги

```
MIN_SEGMENT  — физическое ограничение машины (м)
               Меньше значение → больше гибкости, но мелкие куски физически нельзя
               Больше значение → меньше мусорных отрезков, но могут быть невыполнимые заказы

MAX_SPLITS   — максимум барабанов на один кабель
               1 = ЗАПРЕТИТЬ дробление (0 смен барабанов на кабель!)
               2 = не более 2 барабанов на кабель
               5 = свобода (по умолчанию)

WASTE_WEIGHT — приоритет минимизации отходов vs смен барабанов
               1000 (по умолч.) = сначала минимум отходов, потом смены
               10   = смены важнее, отходы вторичны
               0    = только минимизировать смены, отходы не важны
```

### 3. Сценарии

**Хочу нулевые смены барабанов:**
```python
MAX_SPLITS   = 1    # каждый кабель строго с одного барабана
WASTE_WEIGHT = 0    # отходы не важны
```
⚠️ Отходы вырастут — часть барабана останется неиспользованной.

**Хочу минимум отходов (по умолчанию):**
```python
MAX_SPLITS   = 5
WASTE_WEIGHT = 1000
```

**Баланс: мало смен И мало отходов:**
```python
MAX_SPLITS   = 2    # не более 2 барабанов на кабель
WASTE_WEIGHT = 100  # отходы важны, но смены тоже учитываются
```

**Физически маленький кабель (< MIN_SEGMENT):**  
Алгоритм автоматически ставит его на 1 барабан. Ничего не нужно делать.


In [6]:
import os
from dataclasses import dataclass
from typing import List
from collections import defaultdict
from datetime import datetime
from itertools import permutations
import traceback
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

try:
    from ortools.sat.python import cp_model
    import ortools
    print(f'✓ OR-Tools {ortools.__version__}')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'ortools'])
    from ortools.sat.python import cp_model
    import ortools
    print(f'✓ OR-Tools {ortools.__version__} (установлен)')


# ── Структуры данных ────────────────────────────────────────────────────────

@dataclass
class Drum:
    id: str
    length: int

@dataclass
class Cable:
    name: str
    min_length: int
    cores: int
    colors: List[str]
    flexible: bool = False
    def __post_init__(self):
        if len(self.colors) != self.cores:
            raise ValueError(f'{self.name}: цветов {len(self.colors)} != жил {self.cores}')

@dataclass
class Allocation:
    cable_name: str
    color: str
    drum_id: str
    length: int


# ── FFD bin packing ───────────────────────────────────────────────────────────

def _pack_output_drums(allocations, output_drum_length):
    by_color = defaultdict(list)
    seen = set()
    for a in allocations:
        key = (a.cable_name, a.color, a.drum_id)
        if key not in seen:
            seen.add(key)
            by_color[a.color].append(
                {'cable': a.cable_name, 'src_drum': a.drum_id, 'length': a.length})
    out = {}
    for color, segs in by_color.items():
        segs_s = sorted(segs, key=lambda s: -s['length'])
        drums  = []
        for seg in segs_s:
            placed = False
            for drum in drums:
                if not drum['overflow'] and drum['used'] + seg['length'] <= output_drum_length:
                    drum['segs'].append(seg); drum['used'] += seg['length']; placed = True; break
            if not placed:
                drums.append({'used': seg['length'], 'segs': [seg],
                              'overflow': seg['length'] > output_drum_length})
        out[color] = drums
    return out


def _reorder_segs_by_instruction(out_drums_by_color, allocations):
    """Переупорядочить segs по порядку инструкции (порядок намотки снизу вверх)."""
    for color, dl in out_drums_by_color.items():
        instr_order = []
        seen = set()
        for a in allocations:
            if a.color == color:
                key = (a.cable_name, a.drum_id)
                if key not in seen:
                    seen.add(key)
                    instr_order.append(key)
        for drum_info in dl:
            seg_map = {(s['cable'], s['src_drum']): s for s in drum_info['segs']}
            drum_info['segs'] = [seg_map[k] for k in instr_order if k in seg_map]


# ── Группировка кабелей по набору цветов ─────────────────────────────────────
#
# ЗАЧЕМ: кабели с разными наборами цветов (4ж vs 5ж) дают РАЗНОЕ число
# приёмных барабанов по цветам, поэтому FFD нужно запускать раздельно
# для каждой группы. Иначе «набор» Нi будет содержать провода разных
# цветов из РАЗНЫХ барабанов → скрутить невозможно.

def _group_cables_by_colors(cables):
    """Группирует кабели по frozenset(colors).
    Возвращает список dict (по убыванию числа цветов):
      [{'key', 'label': '5ж'/'4ж', 'colors', 'cables', 'names'}, ...]
    """
    groups = {}
    color_order = {}
    for cable in cables:
        key = frozenset(cable.colors)
        if key not in groups:
            groups[key] = []
            color_order[key] = list(cable.colors)
        groups[key].append(cable)

    result = []
    for key, cabs in sorted(groups.items(), key=lambda x: -len(x[0])):
        result.append({
            'key':    key,
            'label':  f'{len(key)}ж',
            'colors': color_order[key],
            'cables': cabs,
            'names':  {c.name for c in cabs},
        })
    # Disambiguate labels when multiple groups have same core count (e.g. two 4ж groups)
    from collections import Counter
    label_counts = Counter(g['label'] for g in result)
    label_seq    = {}
    SUFFIX = 'АБВГДЕЖЗИК'
    for g in result:
        lbl = g['label']
        if label_counts[lbl] > 1:
            idx = label_seq.get(lbl, 0)
            g['label'] = f'{lbl}-{SUFFIX[idx]}'
            label_seq[lbl] = idx + 1
    return result


def _pack_output_drums_by_group(allocations, output_drum_length, cable_groups):
    """FFD упаковка отдельно для каждой группы кабелей.

    Мутирует каждый group-dict, добавляя:
      out_drums_by_color, n_nabors, overflow, total_drums

    Возвращает out_drum_map: (cable_name, color, src_drum) -> 'Xж-НY'

    Метки наборов: '5ж-Н1', '5ж-Н2', '4ж-Н1', ...
    Приёмный бар.: '5ж-Н1-ж/з', '4ж-Н1-чёрный', ...
    """
    out_drum_map = {}

    for g in cable_groups:
        group_allocs = [a for a in allocations if a.cable_name in g['names']]

        odc = _pack_output_drums(group_allocs, output_drum_length)
        _reorder_segs_by_instruction(odc, group_allocs)

        g['out_drums_by_color'] = odc

        first_color = g['colors'][0]
        n_nabors = len(odc.get(first_color, []))
        g['n_nabors']    = n_nabors
        g['overflow']    = any(d['overflow'] for dl in odc.values() for d in dl)
        g['total_drums'] = sum(len(dl) for dl in odc.values())

        # (cable, src_drum) → № набора внутри группы
        nabor_num_map = {}
        for ni, drum_info in enumerate(odc.get(first_color, [])):
            for seg in drum_info['segs']:
                nabor_num_map[(seg['cable'], seg['src_drum'])] = ni + 1

        for color, dl in odc.items():
            for drum_info in dl:
                for seg in drum_info['segs']:
                    ni = nabor_num_map.get((seg['cable'], seg['src_drum']), 0)
                    out_drum_map[(seg['cable'], color, seg['src_drum'])] = f'{g["label"]}-Н{ni}'

    return out_drum_map


# ── Оптимизация порядка цветов (5! перебор) ──────────────────────────────────

def _best_color_order(allocations_raw, cables, drum_penalty, color_penalty):
    all_colors, seen = [], set()
    for c in cables:
        for col in c.colors:
            if col not in seen: all_colors.append(col); seen.add(col)
    if len(all_colors) <= 1:
        return all_colors, 0
    cable_main = {}
    for a in allocations_raw:
        if a.cable_name not in cable_main or a.length > cable_main[a.cable_name][1]:
            cable_main[a.cable_name] = (a.drum_id, a.length)
    by_col = defaultdict(lambda: defaultdict(list))
    for a in allocations_raw:
        by_col[a.color][a.cable_name].append(a)
    def endpoints(color):
        names = sorted(by_col.get(color, {}).keys(),
                       key=lambda n: cable_main.get(n, ('zzz', 0))[0])
        if not names: return None, None
        fd = sorted(by_col[color][names[0]],  key=lambda a: -a.length)[0].drum_id
        ld = sorted(by_col[color][names[-1]], key=lambda a: -a.length)[-1].drum_id
        return fd, ld
    ep = {col: endpoints(col) for col in all_colors}
    def _route_cost(order):
        cost = (len(order) - 1) * color_penalty
        for i in range(len(order) - 1):
            _, ld = ep[order[i]]; fd, _ = ep[order[i+1]]
            if ld and fd and ld != fd: cost += drum_penalty
        return cost

    # Small sets: exact permutation search (≤8 colors)
    # Large sets: greedy nearest-neighbor (pair cables with 16+ colors)
    if len(all_colors) <= 8:
        best_cost, best_order = float('inf'), all_colors
        for perm in permutations(all_colors):
            cost = _route_cost(list(perm))
            if cost < best_cost: best_cost, best_order = cost, list(perm)
    else:
        # Greedy: start from each color, pick cheapest unvisited next neighbor
        def greedy_from(start):
            remaining = [c for c in all_colors if c != start]
            order = [start]
            while remaining:
                last = order[-1]
                _, ld = ep[last]
                best_next, best_c = None, float('inf')
                for nxt in remaining:
                    fd, _ = ep[nxt]
                    c = color_penalty + (drum_penalty if ld and fd and ld != fd else 0)
                    if c < best_c: best_c, best_next = c, nxt
                order.append(best_next)
                remaining.remove(best_next)
            return order, _route_cost(order)
        best_cost, best_order = float('inf'), all_colors
        for start in all_colors:
            order, cost = greedy_from(start)
            if cost < best_cost: best_cost, best_order = cost, order
    return best_order, best_cost


# ── OR-Tools CP-SAT ───────────────────────────────────────────────────────────

def solve(drums, cables,
          min_segment=350, max_splits=1, waste_weight=1000,
          drum_change_penalty=0, color_change_penalty=0,
          output_drum_length=0, time_limit=60.0):

    splice_warning = max_splits > 1
    model = cp_model.CpModel()
    K, D = len(cables), len(drums)
    all_colors, seen_c = [], set()
    for c in cables:
        for col in c.colors:
            if col not in seen_c: all_colors.append(col); seen_c.add(col)
    C = len(all_colors)

    x, y = {}, {}
    for k, cable in enumerate(cables):
        is_short = cable.min_length < min_segment
        for d in range(D):
            x[k,d] = model.NewIntVar(0, drums[d].length, f'x{k}_{d}')
            y[k,d] = model.NewBoolVar(f'y{k}_{d}')
            model.Add(x[k,d] == 0).OnlyEnforceIf(y[k,d].Not())
            if not is_short: model.Add(x[k,d] >= min_segment).OnlyEnforceIf(y[k,d])
            if output_drum_length > 0: model.Add(x[k,d] <= output_drum_length)
        du = sum(y[k,d] for d in range(D))
        model.Add(du == 1 if is_short else du <= max_splits)
    for k, cable in enumerate(cables):
        t = sum(x[k,d] for d in range(D))
        model.Add(t >= cable.min_length if cable.flexible else t == cable.min_length)
    for d, drum in enumerate(drums):
        model.Add(sum(x[k,d] * cables[k].cores for k in range(K)) <= drum.length)

    total_wire = sum(x[k,d]*cables[k].cores for k in range(K) for d in range(D))
    n_segs     = sum(y[k,d] for k in range(K) for d in range(D))

    if drum_change_penalty > 0:
        w = {}
        for ci, color in enumerate(all_colors):
            cabs = [k for k in range(K) if color in cables[k].colors]
            for d in range(D):
                w[ci,d] = model.NewBoolVar(f'w{ci}_{d}')
                if cabs: model.AddMaxEquality(w[ci,d], [y[k,d] for k in cabs])
                else:    model.Add(w[ci,d] == 0)
        dcp = sum(w[ci,d] for ci in range(C) for d in range(D))
        model.Maximize(waste_weight*total_wire - n_segs - drum_change_penalty*dcp)
    else:
        dcp = None
        model.Maximize(waste_weight*total_wire - n_segs)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.num_search_workers  = 4
    solver.parameters.log_search_progress = False
    status = solver.Solve(model)

    ok = status in (cp_model.OPTIMAL, cp_model.FEASIBLE)
    if not ok:
        return None, None, {'status': solver.StatusName(status), 'error': True,
                            'splice_warning': splice_warning}, None

    total_cap = sum(d.length for d in drums)
    used_wire = sum(solver.Value(x[k,d])*cables[k].cores for k in range(K) for d in range(D))

    allocations_raw, ext_log = [], []
    for k, cable in enumerate(cables):
        segs = sorted([(drums[d].id, solver.Value(x[k,d]))
                       for d in range(D) if solver.Value(x[k,d]) > 0], key=lambda t: -t[1])
        ct = sum(v for _,v in segs)
        for drum_id, meters in segs:
            for color in cable.colors:
                allocations_raw.append(Allocation(cable.name, color, drum_id, meters))
        if cable.flexible and ct > cable.min_length:
            ext_log.append({'Заказ': cable.name, 'Мин. длина (м)': cable.min_length,
                            'Удлинение (м)': ct-cable.min_length, 'Итог (м)': ct,
                            'Жил': cable.cores,
                            'Доп. провод (м)': (ct-cable.min_length)*cable.cores})

    best_color_order, transition_cost = _best_color_order(
        allocations_raw, cables, drum_change_penalty, color_change_penalty)
    allocations = _order_for_instruction(allocations_raw, cables, best_color_order)

    # ── Группировка + FFD по группам ────────────────────────────────────────
    cable_groups   = None
    out_drum_map   = {}
    total_out_drums = nabor_count = 0
    out_overflow   = False
    recv_drum_changes = 0

    if output_drum_length > 0:
        cable_groups = _group_cables_by_colors(cables)
        out_drum_map = _pack_output_drums_by_group(allocations, output_drum_length, cable_groups)
        total_out_drums = sum(g['total_drums'] for g in cable_groups)
        nabor_count     = sum(g['n_nabors']    for g in cable_groups)
        out_overflow    = any(g['overflow']     for g in cable_groups)

        prev_r = None
        for a in allocations:
            r = out_drum_map.get((a.cable_name, a.color, a.drum_id), '')
            if r and prev_r is not None and r != prev_r: recv_drum_changes += 1
            if r: prev_r = r

    dcp_val = solver.Value(dcp) if drum_change_penalty > 0 else None
    drum_ch, color_ch = _count_instruction_changes(allocations)

    stats = {
        'status': solver.StatusName(status), 'error': False,
        'splice_warning': splice_warning,
        'waste': total_cap - used_wire, 'n_segs': solver.Value(n_segs),
        'drum_col_pairs': dcp_val, 'drum_changes': drum_ch, 'color_changes': color_ch,
        'color_order': best_color_order, 'transition_cost': transition_cost,
        'ext_count': len(ext_log), 'absorbed': sum(e['Доп. провод (м)'] for e in ext_log),
        'cable_groups':     cable_groups,
        'nabor_count':      nabor_count,
        'total_out_drums':  total_out_drums,
        'out_overflow':     out_overflow,
        'out_drum_map':     out_drum_map if output_drum_length > 0 else None,
        'recv_drum_changes': recv_drum_changes,
    }
    return allocations, ext_log, stats, solver


# ── Вспомогательные ───────────────────────────────────────────────────────────

def _order_for_instruction(allocations, cables, color_order=None):
    if color_order is None:
        color_order, seen = [], set()
        for c in cables:
            for col in c.colors:
                if col not in seen: color_order.append(col); seen.add(col)
    cable_main = {}
    for a in allocations:
        if a.cable_name not in cable_main or a.length > cable_main[a.cable_name][1]:
            cable_main[a.cable_name] = (a.drum_id, a.length)
    by_col = defaultdict(lambda: defaultdict(list))
    for a in allocations: by_col[a.color][a.cable_name].append(a)
    result = []
    for col in color_order:
        groups = by_col.get(col, {})
        names  = sorted(groups.keys(), key=lambda n: cable_main.get(n, ('zzz',0))[0])
        for name in names:
            result.extend(sorted(groups[name], key=lambda a: -a.length))
    return result

def _count_instruction_changes(allocations):
    dc = cc = 0
    for i in range(1, len(allocations)):
        p, c = allocations[i-1], allocations[i]
        if c.drum_id != p.drum_id: dc += 1
        if c.color   != p.color:   cc += 1
    return dc, cc


# ── Инструкция изолировщика ───────────────────────────────────────────────────

def create_instructions(allocations, drums, cables, out_drum_map=None):
    balances = {d.id: float(d.length) for d in drums}
    rows = []
    prev_src_drum  = {}
    prev_color     = None
    prev_recv_drum = None

    for task in allocations:
        cable = next((c for c in cables if c.name == task.cable_name), None)

        src_changed   = (prev_src_drum.get(task.color) is not None and
                         task.drum_id != prev_src_drum[task.color])
        color_changed = (prev_color is not None and task.color != prev_color)

        nabor_label = recv_label = ''
        recv_changed = False
        if out_drum_map is not None:
            nabor_label  = out_drum_map.get((task.cable_name, task.color, task.drum_id), '')
            recv_label   = f'{nabor_label}-{task.color}' if nabor_label else ''
            recv_changed = (prev_recv_drum is not None and recv_label != prev_recv_drum)

        before = balances[task.drum_id]
        balances[task.drum_id] -= task.length
        after  = balances[task.drum_id]

        row = {
            'Смена отдающего':  '!' if src_changed   else '',
            'Смена цвета':      '!' if color_changed  else '',
            'Смена приёмного':  '!' if recv_changed   else '',
            'Цвет':             task.color,
            'Заказ':            task.cable_name,
            'Можно увеличить?': 'да' if (cable and cable.flexible) else 'нет',
            'Барабан':          task.drum_id,
            'Изолировать (м)':  task.length,
            'Было (м)':         int(before),
            'Осталось (м)':     int(after),
        }
        if out_drum_map is not None:
            row['Приёмный бар.']     = recv_label
            row['Набор для скрутки'] = nabor_label

        rows.append(row)
        prev_src_drum[task.color] = task.drum_id
        prev_color = task.color
        if recv_label:
            prev_recv_drum = recv_label

    df = pd.DataFrame(rows)
    df.insert(0, '№', range(1, len(df) + 1))

    cols_recv = ['№','Смена отдающего','Смена цвета','Смена приёмного',
                 'Цвет','Заказ','Можно увеличить?','Барабан',
                 'Изолировать (м)','Приёмный бар.','Набор для скрутки',
                 'Было (м)','Осталось (м)']
    cols_bare = ['№','Смена отдающего','Смена цвета',
                 'Цвет','Заказ','Можно увеличить?','Барабан',
                 'Изолировать (м)','Было (м)','Осталось (м)']
    target = cols_recv if out_drum_map is not None else cols_bare
    return df[[c for c in target if c in df.columns]]


def create_drum_summary(allocations, drums, cables):
    cores_map = {c.name: c.cores for c in cables}
    used, seen = defaultdict(int), set()
    for a in allocations:
        key = (a.cable_name, a.drum_id)
        if key not in seen:
            seen.add(key); used[a.drum_id] += a.length * cores_map.get(a.cable_name, 1)
    rows = []
    for drum in drums:
        u = used.get(drum.id, 0); r = drum.length - u
        rows.append({'Барабан': drum.id, 'Ёмкость (м)': drum.length,
                     'Использовано (м)': u, 'Остаток (м)': r,
                     'Загрузка': f'{u/drum.length*100:.1f}%'})
    return pd.DataFrame(rows)


# ── Приёмные катушки: таблица по группам и наборам ───────────────────────────

def create_receiving_drums_table(cable_groups, output_drum_length):
    if not cable_groups:
        return pd.DataFrame()
    rows = []
    for g in cable_groups:
        odc    = g.get('out_drums_by_color', {})
        colors = g['colors']
        glabel = g['label']
        for ni in range(g['n_nabors']):
            nabor_label = f'{glabel}-Н{ni+1}'
            for color in colors:
                if ni >= len(odc.get(color, [])):
                    continue
                d = odc[color][ni]
                rows.append({
                    'Группа':          glabel,
                    'Набор':           nabor_label,
                    'Приёмный бар.':   f'{nabor_label}-{color}',
                    'Цвет':            color,
                    'Намотка (м) ↑':   '+'.join(str(s['length']) for s in d['segs']),
                    'Итого (м)':       d['used'],
                    'Ёмкость (м)':     output_drum_length,
                    'Остаток (м)':     output_drum_length - d['used'],
                    'Загрузка':        f'{d["used"]/output_drum_length*100:.1f}%',
                    'Статус':          'ПЕРЕПОЛНЕН' if d['overflow'] else 'OK',
                })
    return pd.DataFrame(rows)


# ── Инструкция скрутки ────────────────────────────────────────────────────────
#
# Порядок скрутки = ОБРАТНЫЙ порядку намотки:
#   изолировщик мотает снизу вверх → скрутчик разматывает сверху вниз

def create_twisting_instruction(cable_groups, cables):
    if not cable_groups:
        return pd.DataFrame()
    cables_map = {c.name: c for c in cables}
    rows, seq  = [], 1
    for g in cable_groups:
        odc         = g.get('out_drums_by_color', {})
        colors      = g['colors']
        glabel      = g['label']
        first_color = colors[0]
        for ni in range(g['n_nabors']):
            nabor_label = f'{glabel}-Н{ni+1}'
            drum_info   = odc[first_color][ni]
            load_str    = ' + '.join(f'{nabor_label}-{c}' for c in colors)
            nabor_total = drum_info['used']
            first_row   = True
            for seg in reversed(drum_info['segs']):   # обратный порядок!
                cable = cables_map.get(seg['cable'])
                rows.append({
                    'Группа':             glabel      if first_row else '',
                    'Набор':              nabor_label if first_row else '',
                    'Загрузить катушки':  load_str    if first_row else '',
                    'Итого в наборе (м)': nabor_total if first_row else '',
                    '№':                  seq,
                    'Кабель':             seg['cable'],
                    'Длина скрутки (м)':  seg['length'],
                    'Жил':                cable.cores if cable else '',
                })
                first_row = False; seq += 1
    cols = ['Группа','Набор','Загрузить катушки','Итого в наборе (м)',
            '№','Кабель','Длина скрутки (м)','Жил']
    if not rows:
        return pd.DataFrame(columns=cols)
    df = pd.DataFrame(rows)
    return df[[c for c in cols if c in df.columns]]


# ── Excel экспорт ─────────────────────────────────────────────────────────────

def export_to_excel(filename, drums, cables, allocations, ext_log,
                    cable_groups=None, output_drum_length=0, out_drum_map=None):

    # Base palette for known colors; unknown colors get auto-assigned from extras
    _COLOR_HEX_BASE = {'ж/з':'FFFACD','синий':'DDEEFF','чёрный':'EBEBEB',
                       'коричневый':'F5E6D3','натуральный':'F8F8F0',
                       'красный':'FFD6D6','белый':'FFFFFF','серый':'E8E8E8',
                       'оранжевый':'FFE8CC','фиолетовый':'EDD6FF',
                       'розовый':'FFD6EC','жёлтый':'FFFAB0',
                       'голубой':'D6F0FF','зелёный':'D6FFD8'}
    _AUTO_PALETTE = ['E8F5E9','FFF3E0','E3F2FD','FCE4EC','F3E5F5',
                     'E0F7FA','FFF8E1','EFEBE9','F9FBE7','E8EAF6']
    all_used_colors = set()
    if cables: all_used_colors = {col for c in cables for col in c.colors}
    COLOR_HEX = dict(_COLOR_HEX_BASE)
    _auto_idx = 0
    for col in sorted(all_used_colors):
        if col not in COLOR_HEX:
            COLOR_HEX[col] = _AUTO_PALETTE[_auto_idx % len(_AUTO_PALETTE)]
            _auto_idx += 1
    SRC_CHANGE_HEX  = 'FF8C00'
    COL_CHANGE_HEX  = 'DAA520'
    RECV_CHANGE_HEX = 'A9A9A9'
    # Цвета групп для листа катушек и скрутки
    GROUP_FILL_HEX  = ['D6E4F0', 'FFF0D6', 'D6F0D8', 'F0D6F0']  # ≤4 группы

    def fill(h): return PatternFill(start_color=h, end_color=h, fill_type='solid')
    thin = Border(left=Side(style='thin'),  right=Side(style='thin'),
                  top=Side(style='thin'),   bottom=Side(style='thin'))
    hdr_fill = fill('D3D3D3')

    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        df_d = pd.DataFrame([{'Барабан': d.id, 'Ёмкость (м)': d.length} for d in drums])
        df_c = pd.DataFrame([{'Заказ': c.name, 'Мин. длина (м)': c.min_length,
                               'Жил': c.cores, 'Цвета': ', '.join(c.colors),
                               'Можно увеличить?': 'да' if c.flexible else 'нет'} for c in cables])
        df_d.to_excel(writer, sheet_name='1. Данные', startrow=1, index=False)
        df_c.to_excel(writer, sheet_name='1. Данные', startrow=len(df_d)+5, index=False)
        writer.sheets['1. Данные']['A1'] = 'БАРАБАНЫ (ИСТОЧНИКИ)'
        writer.sheets['1. Данные'][f'A{len(df_d)+5}'] = 'ЗАКАЗЫ'

        if ext_log:
            pd.DataFrame(ext_log).to_excel(writer, sheet_name='2. Удлинения', startrow=1, index=False)
            writer.sheets['2. Удлинения']['A1'] = 'ГИБКИЕ ЗАКАЗЫ — УДЛИНЕНИЯ'

        create_instructions(allocations, drums, cables, out_drum_map=out_drum_map)\
            .to_excel(writer, sheet_name='3. Инструкция (изол.)', startrow=1, index=False)
        writer.sheets['3. Инструкция (изол.)']['A1'] = 'ИНСТРУКЦИЯ — ИЗОЛИРОВЩИК'

        create_drum_summary(allocations, drums, cables)\
            .to_excel(writer, sheet_name='4. Источники (остатки)', startrow=1, index=False)
        writer.sheets['4. Источники (остатки)']['A1'] = 'ОСТАТКИ НА БАРАБАНАХ-ИСТОЧНИКАХ'

        if cable_groups and output_drum_length > 0:
            create_receiving_drums_table(cable_groups, output_drum_length)\
                .to_excel(writer, sheet_name='5. Приёмные катушки', startrow=1, index=False)
            writer.sheets['5. Приёмные катушки']['A1'] = (
                f'ПРИЁМНЫЕ КАТУШКИ — {output_drum_length} м/жилу '
                f'| Намотка ↑ = снизу вверх  |  Скрутка = сверху вниз')

            create_twisting_instruction(cable_groups, cables)\
                .to_excel(writer, sheet_name='6. Скрутка', startrow=1, index=False)
            writer.sheets['6. Скрутка']['A1'] = (
                'ИНСТРУКЦИЯ СКРУТКИ — по группам и наборам '
                '| порядок = обратный намотке (верх катушки первым)')

    wb = load_workbook(filename)

    # ── Общий стиль ──────────────────────────────────────────────────
    for sn in wb.sheetnames:
        ws = wb[sn]
        try:
            for col in ws.columns:
                w = max((len(str(c.value)) for c in col if c.value), default=8)
                ws.column_dimensions[col[0].column_letter].width = min(w + 3, 55)
            for cell in ws[2]:
                if cell.value:
                    cell.font = Font(bold=True, size=11); cell.fill = hdr_fill
                    cell.alignment = Alignment(horizontal='center', vertical='center')
                    cell.border = thin
            for row in ws.iter_rows():
                for cell in row:
                    if cell.value is not None:
                        cell.border = thin
                        cell.alignment = Alignment(vertical='center', wrap_text=True)
        except Exception as e:
            print(f'  Предупреждение: общий стиль "{sn}": {e}')

    # ── Стиль: инструкция изолировщика ───────────────────────────────
    try:
        ws = wb['3. Инструкция (изол.)']
        ci = {cell.value: cell.column - 1 for cell in ws[2] if cell.value}
        i_src  = ci.get('Смена отдающего')
        i_col  = ci.get('Смена цвета')
        i_recv = ci.get('Смена приёмного')
        i_tsv  = ci.get('Цвет', 0)
        for row in ws.iter_rows(min_row=3):
            cv = row[i_tsv].value if i_tsv < len(row) else None
            if cv in COLOR_HEX:
                rf = fill(COLOR_HEX[cv])
                for cell in row:
                    if cell.value is not None: cell.fill = rf
            for idx, hex_c in [(i_src, SRC_CHANGE_HEX), (i_col, COL_CHANGE_HEX),
                                (i_recv, RECV_CHANGE_HEX)]:
                if idx is not None and idx < len(row) and row[idx].value == '!':
                    c = row[idx]
                    c.fill = fill(hex_c); c.font = Font(bold=True, size=11, color='FFFFFF')
                    c.alignment = Alignment(horizontal='center', vertical='center')
    except Exception as e:
        print(f'  Предупреждение: стиль листа изолировщика: {e}')

    # ── Стиль: приёмные катушки ───────────────────────────────────────
    try:
        if '5. Приёмные катушки' in wb.sheetnames:
            ws = wb['5. Приёмные катушки']
            ci  = {cell.value: cell.column - 1 for cell in ws[2] if cell.value}
            gi  = ci.get('Группа', 0)
            ni  = ci.get('Набор',  1)
            ti  = ci.get('Цвет',   3)
            sti = ci.get('Статус')
            # Карта группа → цвет заливки
            group_fills = {}
            gfi = 0
            for row in ws.iter_rows(min_row=3):
                gv  = row[gi].value  if gi  < len(row) else None
                nv  = row[ni].value  if ni  < len(row) else None
                cv  = row[ti].value  if ti  < len(row) else None
                stv = row[sti].value if (sti is not None and sti < len(row)) else None
                if gv and gv not in group_fills:
                    group_fills[gv] = GROUP_FILL_HEX[gfi % len(GROUP_FILL_HEX)]; gfi += 1
                gf = group_fills.get(gv, 'FFFFFF') if gv else None
                if stv and 'ПЕРЕПОЛНЕН' in str(stv):
                    for cell in row:
                        if cell.value is not None: cell.fill = fill('FFCCCC'); cell.font = Font(bold=True)
                elif cv in COLOR_HEX:
                    for cell in row:
                        if cell.value is not None: cell.fill = fill(COLOR_HEX[cv])
                if nv:
                    row[ni].font = Font(bold=True, size=11)
                if gv and gi < len(row) and row[gi].value:
                    row[gi].fill = fill(gf); row[gi].font = Font(bold=True, size=11)
    except Exception as e:
        print(f'  Предупреждение: стиль листа катушек: {e}')

    # ── Стиль: скрутка ────────────────────────────────────────────────
    try:
        if '6. Скрутка' in wb.sheetnames:
            ws = wb['6. Скрутка']
            ci = {cell.value: cell.column - 1 for cell in ws[2] if cell.value}
            gi = ci.get('Группа', 0)
            ni = ci.get('Набор',  1)
            group_fills = {}
            gfi = 0
            for row in ws.iter_rows(min_row=3):
                gv = row[gi].value if gi < len(row) else None
                nv = row[ni].value if ni < len(row) else None
                if gv and gv not in group_fills:
                    group_fills[gv] = GROUP_FILL_HEX[gfi % len(GROUP_FILL_HEX)]; gfi += 1
                if nv:   # строка-заголовок набора
                    gf = group_fills.get(gv or '', 'E8E0F0')
                    for cell in row:
                        if cell.value is not None:
                            cell.fill = fill(gf)
                            cell.font = Font(bold=True, size=11)
                            cell.alignment = Alignment(vertical='center', wrap_text=True)
    except Exception as e:
        print(f'  Предупреждение: стиль листа скрутки: {e}')

    wb.save(filename)
    return filename


print('✓ v4.0 | раздельный FFD по группам жильности (5ж / 4ж / ...)')
print('  Набор Xж-НY содержит только кабели одной группы → скрутка корректна')


✓ OR-Tools 9.15.6755
✓ v4.0 | раздельный FFD по группам жильности (5ж / 4ж / ...)
  Набор Xж-НY содержит только кабели одной группы → скрутка корректна


---
## РАЗДЕЛ 2: ИСХОДНЫЕ ДАННЫЕ

In [7]:
# ЗАпуск 5*16
drums = [
    Drum(id='Барабан-1', length=10000),
    Drum(id='Барабан-2', length=10000),
    Drum(id='Барабан-3', length=10000),
    Drum(id='Барабан-4', length=11000),
    Drum(id='Барабан-5', length=9931),
]
# ЗАпуск 4*16

# drums = [
#     Drum(id='Барабан-1', length=1000),
#     Drum(id='Барабан-2', length=10000),
#     Drum(id='Барабан-3', length=3730),
#     Drum(id='Барабан-4', length=4925),
#     # Drum(id='Барабан-5', length=9931),
# ]

print('БАРАБАНЫ:')
for d in drums:
    print(f'  {d.id}: {d.length} м')
print(f'\nИтого: {sum(d.length for d in drums)} м')


БАРАБАНЫ:
  Барабан-1: 10000 м
  Барабан-2: 10000 м
  Барабан-3: 10000 м
  Барабан-4: 11000 м
  Барабан-5: 9931 м

Итого: 50931 м


In [8]:
# ══════════════════════════════════════════════════════════════════════
# min_length — ЦЕЛОЕ ЧИСЛО метров на жилу
# Примечание: 705×1.1=775.5 → 776; 1035×1.1=1138.5 → 1139 (округление вверх)
#
# flexible=False  →  "сдача ОДНОЙ длиной"  — строго фиксировано
# flexible=True   →  "сдача МАКС. длиной"  — алгоритм УВЕЛИЧИТ длину
# ══════════════════════════════════════════════════════════════════════

C5 = ['ж/з', 'синий', 'чёрный', 'коричневый', 'натуральный']  # 5-жильный
C4 = ['ж/з', 'чёрный', 'коричневый', 'натуральный']          # 4-жильный

cables = [
    Cable('п.10 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 460м*1,1 одной длиной',   int(460*1.1) , 5, C5, flexible=False),
    Cable('п.30 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 430м*1,1 одной длиной',   int(430*1.1) , 5, C5, flexible=False),
    Cable('п.31 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 670м*1,1 макс. длиной',   int(670*1.1) , 5, C5, flexible=False),
    Cable('п.37 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 790м*1,1 макс. длиной',   int(790*1.1) , 5, C5, flexible=False),
    Cable('п.42 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 350м*1,1 одной длиной',   int(350*1.1) , 5, C5, flexible=False),
    Cable('п.51 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 705м*1,1 одной длиной',   int(705*1.1) , 5, C5, flexible=False),
    Cable('п.53 КОСМОГЕР Вз-ВВГнг(А)-LS 5х16мк(N,PE)-1, 400м*1,1 одной длиной',   int(400*1.1) , 5, C5, flexible=False),
    Cable('п.56 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 1035м*1,1 одной длиной',   int(1035*1.1) , 5, C5, flexible=False),
    Cable('п.88 КОСМОГЕР Вз-ВВГнг(А)-LS 5х16мк(N,PE)-1, 120м*1,1 одной длиной',   int(120*1.1) , 5, C5, flexible=False),
    Cable('п.14/1 КОСМОГЕР Вз-ВБШвнг(PE)-LS 4х16мк(N)-1, 1000м',   1000 , 4, C4, flexible=True),
    Cable('п.14/2 КОСМОГЕР Вз-ВБШвнг(PE)-LS 4х16мк(N)-1, 1000м',   1000 , 4, C4, flexible=True),
    Cable('п.14/3 КОСМОГЕР Вз-ВБШвнг(PE)-LS 4х16мк(N)-1, 1000м',   1000 , 4, C4, flexible=True),
    Cable('п.14/4 КОСМОГЕР Вз-ВБШвнг(PE)-LS 4х16мк(N)-1, 1000м',   1000 , 4, C4, flexible=True),
]

# Запуск 5*16
# cables = [
#     # Cable('п.10 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 460м*1,1 одной длиной',   int(460*1.1) , 5, C5, flexible=False),
#     # Cable('п.30 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 430м*1,1 одной длиной',   int(430*1.1) , 5, C5, flexible=False),
#     Cable('п.10+30+31 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 670м*1,1 макс. длиной',   int(460*1.1)+int(430*1.1)+int(670*1.1) , 5, C5, flexible=False),
#     # Cable('п.37 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 790м*1,1 макс. длиной',   int(790*1.1) , 5, C5, flexible=False),
#     Cable('п.37+42 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 350м*1,1 одной длиной',   int(790*1.1)+int(350*1.1) , 5, C5, flexible=False),
#     # Cable('п.51 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 705м*1,1 одной длиной',   int(705*1.1) , 5, C5, flexible=False),
#     Cable('п.51+53 КОСМОГЕР Вз-ВВГнг(А)-LS 5х16мк(N,PE)-1, 400м*1,1 одной длиной',   int(705*1.1)+int(400*1.1) , 5, C5, flexible=False),
#     # Cable('п.56 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 1035м*1,1 одной длиной',   int(1035*1.1) , 5, C5, flexible=False),
#     Cable('п.56+88 КОСМОГЕР Вз-ВВГнг(А)-LS 5х16мк(N,PE)-1, 120м*1,1 одной длиной',   int(1035*1.1)+int(120*1.1) , 5, C5, flexible=False),
# ]

# Запуск 4*16

# cables = [
#     Cable('п.14/1 КОСМОГЕР Вз-ВБШвнг(PE)-LS 4х16мк(N)-1, 1000м',   1000 , 4, C4, flexible=True),
#     Cable('п.14/2 КОСМОГЕР Вз-ВБШвнг(PE)-LS 4х16мк(N)-1, 1000м',   1000 , 4, C4, flexible=True),
#     Cable('п.14/3 КОСМОГЕР Вз-ВБШвнг(PE)-LS 4х16мк(N)-1, 1000м',   1000 , 4, C4, flexible=True),
#     Cable('п.14/4 КОСМОГЕР Вз-ВБШвнг(PE)-LS 4х16мк(N)-1, 1000м',   1000 , 4, C4, flexible=True),
# ]


total_min   = sum(c.min_length * c.cores for c in cables)
total_avail = sum(d.length for d in drums)
fixed_c     = [c for c in cables if not c.flexible]
flex_c      = [c for c in cables if c.flexible]

print(f'Фиксированных: {len(fixed_c)}')
for c in fixed_c:
    print(f'  {c.name:50}  {c.min_length:5}м × {c.cores}ж = {c.min_length*c.cores:6}м')

print(f'\nГибких:        {len(flex_c)}')
for c in flex_c:
    print(f'  {c.name:50}  {c.min_length:5}м+× {c.cores}ж = {c.min_length*c.cores:6}м')

print(f'\nМинимум нужно:  {total_min} м')
print(f'Доступно:       {total_avail} м')
print(f'Теор. отходы:   {total_avail - total_min} м  ← идеальный минимум')


Фиксированных: 9
  п.10 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 460м*1,1 одной длиной    506м × 5ж =   2530м
  п.30 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 430м*1,1 одной длиной    473м × 5ж =   2365м
  п.31 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 670м*1,1 макс. длиной    737м × 5ж =   3685м
  п.37 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 790м*1,1 макс. длиной    869м × 5ж =   4345м
  п.42 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 350м*1,1 одной длиной    385м × 5ж =   1925м
  п.51 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 705м*1,1 одной длиной    775м × 5ж =   3875м
  п.53 КОСМОГЕР Вз-ВВГнг(А)-LS 5х16мк(N,PE)-1, 400м*1,1 одной длиной    440м × 5ж =   2200м
  п.56 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 1035м*1,1 одной длиной   1138м × 5ж =   5690м
  п.88 КОСМОГЕР Вз-ВВГнг(А)-LS 5х16мк(N,PE)-1, 120м*1,1 одной длиной    132м × 5ж =    660м

Гибких:        4
  п.14/1 КОСМОГЕР Вз-ВБШвнг(PE)-LS 4х16мк(N)-1, 1000м   1000м+× 4ж =   4000м
  п.14/2 КОСМОГЕР Вз-ВБШвнг(PE)-LS 4х16мк(N)-1, 1000

---
## РАЗДЕЛ 3: ЗАПУСК

In [9]:
# ══════════════════════════════════════════════════════════════════════
#  НАСТРОЙКИ — ТОЛЬКО ЗДЕСЬ МЕНЯТЬ
# ══════════════════════════════════════════════════════════════════════

MIN_SEGMENT          = 330    # Физ. минимум сегмента на источнике (м)
MAX_SPLITS           = 5      # 1 = без стыков; >1 = допустить стыки (нарушение ТУ!)
WASTE_WEIGHT         = 2000   # Приоритет минимизации отходов (0…2000)
DRUM_CHANGE_PENALTY  = 500    # Стоимость смены барабана-источника
COLOR_CHANGE_PENALTY = 100    # Стоимость смены цвета в инструкции
OUTPUT_DRUM_LENGTH   = 5000   # Ёмкость приёмной катушки (м/жилу); 0 = не считать
TIME_LIMIT           = 30.0   # Лимит времени решателя (с)

# ══════════════════════════════════════════════════════════════════════

print('=' * 65)
print('  OR-Tools CP-SAT — ПОИСК ОПТИМАЛЬНОГО РЕШЕНИЯ')
print('=' * 65)
print(f'  Мин. сегмент:          {MIN_SEGMENT} м')
if MAX_SPLITS > 1:
    print(f'  Макс. барабанов:  ⚠️  {MAX_SPLITS}  ← СТЫКИ ВНУТРИ КАБЕЛЯ (нарушение ТУ!)')
else:
    print(f'  Макс. барабанов:       {MAX_SPLITS}  (без стыков ✓)')
print(f'  Приоритет отходов:     {WASTE_WEIGHT}')
print(f'  Штраф смены барабана:  {DRUM_CHANGE_PENALTY}')
print(f'  Штраф смены цвета:     {COLOR_CHANGE_PENALTY}')
if OUTPUT_DRUM_LENGTH > 0:
    print(f'  Приёмная катушка:      {OUTPUT_DRUM_LENGTH} м/жилу → наборы Xж-НY')
else:
    print(f'  Приёмная катушка:      не задана')
print(f'  Лимит времени:         {TIME_LIMIT:.0f} с')
print()

allocations, ext_log, stats, solver_obj = solve(
    drums, cables,
    min_segment=MIN_SEGMENT,
    max_splits=MAX_SPLITS,
    waste_weight=WASTE_WEIGHT,
    drum_change_penalty=DRUM_CHANGE_PENALTY,
    color_change_penalty=COLOR_CHANGE_PENALTY,
    output_drum_length=OUTPUT_DRUM_LENGTH,
    time_limit=TIME_LIMIT,
)

print(f'  Статус: {stats["status"]}')
print()

if stats.get('error'):
    print('❌ Решение не найдено!')
else:
    waste = stats['waste']
    theor = sum(d.length for d in drums) - sum(c.min_length * c.cores for c in cables)
    extra = waste - theor

    print('=' * 65)
    print('  РЕЗУЛЬТАТ')
    print('=' * 65)
    if waste == theor:
        print(f'  Отходы источников:  {waste:>6} м  (теор. минимум {theor} м  ✓)')
    else:
        print(f'  Отходы источников:  {waste:>6} м  (теор. минимум {theor} м, доп. +{extra} м)')

    print()
    print(f'  Порядок цветов: {" → ".join(stats["color_order"])}')
    print()
    print(f'  Смен БАРАБАНА (источник):  {stats["drum_changes"]:>3}')
    print(f'  Смен ЦВЕТА:                {stats["color_changes"]:>3}')
    if OUTPUT_DRUM_LENGTH > 0:
        print(f'  Смен НАБОРА (приёмные):    {stats["recv_drum_changes"]:>3}')

    if OUTPUT_DRUM_LENGTH > 0 and stats.get('cable_groups'):
        cable_groups = stats['cable_groups']
        print()
        print('=' * 65)
        print(f'  ПРИЁМНЫЕ КАТУШКИ  (ёмкость {OUTPUT_DRUM_LENGTH} м/жилу)')
        print('=' * 65)
        total_n = stats['nabor_count']
        total_p = stats['total_out_drums']
        print(f'  Итого наборов: {total_n}  ({total_p} физ. катушек)')
        if stats['out_overflow']:
            print('  !! ЕСТЬ ПЕРЕПОЛНЕННЫЕ КАТУШКИ — увеличьте OUTPUT_DRUM_LENGTH')

        for g in cable_groups:
            odc         = g.get('out_drums_by_color', {})
            first_color = g['colors'][0]
            n_col       = len(g['colors'])
            glabel      = g['label']
            n_nab       = g['n_nabors']
            n_phys      = g['total_drums']
            print()
            print(f'  ── Группа {glabel} ({n_col}-жильные): {n_nab} набора × {n_col} цветов = {n_phys} физ. катушек')
            for ni, drum_info in enumerate(odc.get(first_color, [])):
                nlabel   = f'{glabel}-Н{ni+1}'
                namotka  = '+'.join(str(s['length']) for s in drum_info['segs'])
                pct      = drum_info['used'] / OUTPUT_DRUM_LENGTH * 100
                flag     = '  !! ПЕРЕПОЛНЕН' if drum_info['overflow'] else ''
                n_cables = len(drum_info['segs'])
                print(f'    {nlabel}: {n_cables} кабелей, {drum_info["used"]}м [{pct:.0f}%]  {namotka}{flag}')

        print()
        print('  Скрутка (порядок = обратный намотке):')
        seq = 1
        for g in cable_groups:
            odc         = g.get('out_drums_by_color', {})
            first_color = g['colors'][0]
            colors      = g['colors']
            glabel      = g['label']
            print(f'  ─ Группа {glabel} ({len(colors)}-жильные):')
            for ni, drum_info in enumerate(odc.get(first_color, [])):
                nlabel = f'{glabel}-Н{ni+1}'
                load   = ' + '.join(f'{nlabel}-{c}' for c in colors)
                print(f'    Загрузить {load} → крутить:')
                for seg in reversed(drum_info['segs']):
                    print(f'      #{seq}: {seg["cable"]}  {seg["length"]} м')
                    seq += 1

    print()
    if ext_log:
        print(f'  Удлинено заказов: {stats["ext_count"]}  (поглощено {stats["absorbed"]} м)')
    if stats['drum_changes'] > 10:
        print('  >> Много смен барабана. Увеличьте DRUM_CHANGE_PENALTY.')
    elif stats['drum_changes'] <= 4:
        print('  OK: хороший результат по сменам источников.')


  OR-Tools CP-SAT — ПОИСК ОПТИМАЛЬНОГО РЕШЕНИЯ
  Мин. сегмент:          330 м
  Макс. барабанов:  ⚠️  5  ← СТЫКИ ВНУТРИ КАБЕЛЯ (нарушение ТУ!)
  Приоритет отходов:     2000
  Штраф смены барабана:  500
  Штраф смены цвета:     100
  Приёмная катушка:      5000 м/жилу → наборы Xж-НY
  Лимит времени:         30 с

  Статус: OPTIMAL

  РЕЗУЛЬТАТ
  Отходы источников:       0 м  (теор. минимум 7656 м, доп. +-7656 м)

  Порядок цветов: ж/з → синий → чёрный → коричневый → натуральный

  Смен БАРАБАНА (источник):   34
  Смен ЦВЕТА:                  4
  Смен НАБОРА (приёмные):     44

  ПРИЁМНЫЕ КАТУШКИ  (ёмкость 5000 м/жилу)
  Итого наборов: 4  (18 физ. катушек)

  ── Группа 5ж (5-жильные): 2 набора × 5 цветов = 10 физ. катушек
    5ж-Н1: 7 кабелей, 4938м [99%]  869+775+506+737+440+473+1138
    5ж-Н2: 2 кабелей, 517м [10%]  385+132

  ── Группа 4ж (4-жильные): 2 набора × 4 цветов = 8 физ. катушек
    4ж-Н1: 4 кабелей, 5000м [100%]  1000+1500+531+1969
    4ж-Н2: 2 кабелей, 914м [18%]  445+469



---
## РАЗДЕЛ 4: ДЕТАЛЬНЫЕ РЕЗУЛЬТАТЫ

In [10]:
# ── Комплектность ────────────────────────────────────────────────────
print('=' * 65)
print('  КОМПЛЕКТНОСТЬ ЗАКАЗОВ')
print('=' * 65)

ok_n = fail_n = 0
for cable in cables:
    tag = '↑ГИБК' if cable.flexible else ' фикс'
    print(f'[{tag}]  {cable.name}')
    all_ok = True
    for color in cable.colors:
        got = sum(a.length for a in allocations
                  if a.cable_name == cable.name and a.color == color)
        if got >= cable.min_length:
            print(f'    {color:15} ✅ {got} м')
        elif got > 0:
            print(f'    {color:15} ⚠️  {got} / {cable.min_length} м (неполный)')
            all_ok = False
        else:
            print(f'    {color:15} ❌ НЕ ВЫДЕЛЕН (нужно {cable.min_length} м)')
            all_ok = False
    status_str = '✅ КОМПЛЕКТНЫЙ' if all_ok else '❌ НЕКОМПЛЕКТНЫЙ'
    print(f'    → {status_str}')
    if all_ok: ok_n += 1
    else:       fail_n += 1
    print()

print(f'  ✅ Комплектных: {ok_n}/{len(cables)}    ❌ Некомплектных: {fail_n}/{len(cables)}')
print()

# ── Остатки барабанов ────────────────────────────────────────────────
print('=' * 65)
print('  ОСТАТКИ НА БАРАБАНАХ')
print('=' * 65)
df_r = create_drum_summary(allocations, drums, cables)
print(df_r.to_string(index=False))
print(f'\n  ИТОГО ОСТАТОК: {df_r["Остаток (м)"].sum()} м')
print()

# ── Удлинения ────────────────────────────────────────────────────────
print('=' * 65)
print('  ГИБКИЕ ЗАКАЗЫ — УДЛИНЕНИЯ')
print('=' * 65)
if ext_log:
    print(pd.DataFrame(ext_log).to_string(index=False))
else:
    print('  Нет удлинений.')


  КОМПЛЕКТНОСТЬ ЗАКАЗОВ
[ фикс]  п.10 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 460м*1,1 одной длиной
    ж/з             ✅ 506 м
    синий           ✅ 506 м
    чёрный          ✅ 506 м
    коричневый      ✅ 506 м
    натуральный     ✅ 506 м
    → ✅ КОМПЛЕКТНЫЙ

[ фикс]  п.30 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 430м*1,1 одной длиной
    ж/з             ✅ 473 м
    синий           ✅ 473 м
    чёрный          ✅ 473 м
    коричневый      ✅ 473 м
    натуральный     ✅ 473 м
    → ✅ КОМПЛЕКТНЫЙ

[ фикс]  п.31 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 670м*1,1 макс. длиной
    ж/з             ✅ 737 м
    синий           ✅ 737 м
    чёрный          ✅ 737 м
    коричневый      ✅ 737 м
    натуральный     ✅ 737 м
    → ✅ КОМПЛЕКТНЫЙ

[ фикс]  п.37 КОСМОГЕР Вз-ВБШвнг(А)-LS 5х16мк(N,PE)-1, 790м*1,1 макс. длиной
    ж/з             ✅ 869 м
    синий           ✅ 869 м
    чёрный          ✅ 869 м
    коричневый      ✅ 869 м
    натуральный     ✅ 869 м
    → ✅ КОМПЛЕКТНЫЙ

[ фикс]  п.42 КОСМОГЕР В

---
## РАЗДЕЛ 5: ЭКСПОРТ В EXCEL

In [11]:
import traceback as _tb

ts       = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'производственная_инструкция_{ts}.xlsx'

if stats.get('error'):
    print('❌ Нет решения — экспорт пропущен.')
else:
    try:
        export_to_excel(
            filename, drums, cables, allocations, ext_log,
            cable_groups      = stats.get('cable_groups'),
            output_drum_length = OUTPUT_DRUM_LENGTH,
            out_drum_map      = stats.get('out_drum_map'),
        )
        print(f'✓  {filename}')
        print(f'   {os.path.abspath(filename)}')
        print()
        print('  Листы:')
        print('  1. Данные                -- барабаны-источники и заказы')
        if ext_log:
            print('  2. Удлинения             -- гибкие заказы, доп. провод')
        print('  3. Инструкция (изол.)    -- для изолировщика')
        print('     "Смена отдающего" (оранж) | "Смена цвета" (золото) | "Смена приёмного" (серый)')
        print('     "Приёмный бар." = Xж-НY-цвет  |  "Набор для скрутки" = Xж-НY')
        print('  4. Источники (остатки)   -- что осталось на барабанах-источниках')
        if OUTPUT_DRUM_LENGTH > 0 and stats.get('cable_groups'):
            cg  = stats['cable_groups']
            grp = '  +  '.join(f'{g["label"]}: {g["n_nabors"]} наб. × {len(g["colors"])} цв.' for g in cg)
            print(f'  5. Приёмные катушки      -- {grp}  ({stats["total_out_drums"]} физ. катушек)')
            print(f'     "Намотка ↑" = порядок изолирования (снизу вверх)')
            print(f'  6. Скрутка               -- по группам и наборам (порядок обратный намотке)')

    except PermissionError:
        print('⚠️  Файл занят — закройте Excel и повторите.')
    except Exception:
        print('❌ Ошибка при экспорте:')
        _tb.print_exc()


✓  производственная_инструкция_20260315_173015.xlsx
   c:\Users\user\Projects\promtech\test_temp_files\производственная_инструкция_20260315_173015.xlsx

  Листы:
  1. Данные                -- барабаны-источники и заказы
  2. Удлинения             -- гибкие заказы, доп. провод
  3. Инструкция (изол.)    -- для изолировщика
     "Смена отдающего" (оранж) | "Смена цвета" (золото) | "Смена приёмного" (серый)
     "Приёмный бар." = Xж-НY-цвет  |  "Набор для скрутки" = Xж-НY
  4. Источники (остатки)   -- что осталось на барабанах-источниках
  5. Приёмные катушки      -- 5ж: 2 наб. × 5 цв.  +  4ж: 2 наб. × 4 цв.  (18 физ. катушек)
     "Намотка ↑" = порядок изолирования (снизу вверх)
  6. Скрутка               -- по группам и наборам (порядок обратный намотке)
